In [1]:
# !pip install cupy-cuda12x
!pip install cupy-cuda12x scipy statsmodels --upgrade


In [2]:
import cupy as cp
import numpy as np
import time 

### Задание 1  
На вход функции подаётся массив размерностью $N$. Необходимо реализовать вычисление выражения $y = \sqrt{x}+sin(x)⋅exp(−x)$ (используйте встроенные векторные функции, обычный for не даст вам ускорения). Для экспериментов ограничим значение, которое может принимать $N$ в задачах следующим интервалом [1, 10000000].

In [3]:
def pretty_load(i, I, text="", bar_len = 50):
    l = round(i / I * bar_len)
    if text == "":
        text = round(i / I * 100)
    print(f"\r[{'*' * l}{' ' * (bar_len-l)}] {text}", end = "")



In [4]:
def compete1(N_mas, lib_mas, func_mas):
    mmas = []
    for i, N in enumerate(N_mas):
        mmas.append([])
        for j, lib in enumerate(lib_mas):
            pretty_load(i*len(lib_mas) + j, len(N_mas) * len(lib_mas))
            arr1 = lib.arange(N)
            vec_func = lib.vectorize(func_mas[j])
            start = time.time()
            result = vec_func(arr1) 
            mmas[-1].append(time.time() - start)
    pretty_load(len(N_mas) * len(lib_mas), len(N_mas) * len(lib_mas))
    return mmas

def func1_np(x):
    return np.sqrt(x) + np.sin(x)*np.exp(-x)

def func1_cp(x):
    return cp.sqrt(x) + cp.sin(x)*cp.exp(-x)

N_mas = [310000, 1000000, 3100000, 10000000, 31000000]
res = compete1(N_mas, [np, cp], [func1_np, func1_cp])

print(f"""\n{"N count":<12}\t{"np":<12}\t{"cp":<12}""")
for i, row in enumerate(res):
    print(f"{N_mas[i]:<12}\t{row[0]:<12.9f}\t{row[1]:<12.9f}")

[**************************************************] 100
N count     	np          	cp          
310000      	0.928806305 	1.751577854 
1000000     	2.998473406 	0.001640081 
3100000     	8.951680660 	0.002743006 
10000000    	28.516199112	0.002131224 
31000000    	88.361866236	0.004099369 


### Задание 2.

Расчет статистик.

На вход функции подаётся квадртаная матрицы $A$ размерностью $N \times N$. Необходимо реализовать функцию рассчета статистик (среднее значение, медиана, мода, дисперсия, среднеквадартаичное отклонение, первый и третий квартиль) для каждой колонки матрицы. Для экспериментов ограничим значение, которое может принимать $N$ в задачах следующим интервалом [1, 100000].

In [5]:
def compete2(N_mas, lib_mas):
    mmas = []
    for i, N in enumerate(N_mas):
        mmas.append([])
        for j, lib in enumerate(lib_mas):

            pretty_load(i*len(lib_mas) + j, len(N_mas) * len(lib_mas))

            arr1 = lib.random.rand(N, N)
            start = time.time()
            mean        = lib.mean(arr1, axis=1)
            median      = lib.median(arr1, axis=1)
            # mode        = mode_axis(arr1, axis=1)
            var         = lib.var(arr1, axis=1)
            std         = lib.std(arr1, axis=1)
            percentile_25         = lib.percentile(arr1, 25, axis=1)
            percentile_75         = lib.percentile(arr1, 75, axis=1)
            mmas[-1].append(time.time() - start)
    pretty_load(len(N_mas) * len(lib_mas), len(N_mas) * len(lib_mas))
    return mmas

N_mas = [5000, 10000, 15000, 17000]
res = compete2(N_mas, [np, cp])

print(f"""\n{"N count":<12}\t{"np":<12}\t{"cp":<12}""")
for i, row in enumerate(res):
    print(f"{N_mas[i]:<12}\t{row[0]:<12.9f}\t{row[1]:<12.9f}")

[**************************************************] 100
N count     	np          	cp          
5000        	1.187164545 	0.728002787 
10000       	4.648309946 	1.068241119 
15000       	20.064842224	13.964292765
17000       	29.468145370	27.011286497


### Задание 3.

Матричное умножение.

На вход функции подаётся две квадртаная матрицы $A$ и $B$ размерностью $N \times N$. Необходимо реализовать вычисление их произведения $C = A \cdot B$. Для экспериментов ограничим значение, которое может принимать $N$ в задачах следующим интервалом [1, 4096].

In [6]:
def compete3(N_mas, lib_mas):
    mmas = []
    for i, N in enumerate(N_mas):
        mmas.append([])
        for j, lib in enumerate(lib_mas):

            pretty_load(i*len(lib_mas) + j, len(N_mas) * len(lib_mas))

            arr1 = lib.random.rand(N, N)
            arr2 = lib.random.rand(N, N)
            start = time.time()
            res = lib.matmul(arr1, arr2)
            mmas[-1].append(time.time() - start)
    pretty_load(len(N_mas) * len(lib_mas), len(N_mas) * len(lib_mas))
    return mmas

N_mas = [2048, 4096, 8192, 10000]
res = compete3(N_mas, [np, cp])

print(f"""\n{"N count":<12}\t{"np":<12}\t{"cp":<12}""")
for i, row in enumerate(res):
    print(f"{N_mas[i]:<12}\t{row[0]:<12.9f}\t{row[1]:<12.9f}")

[**************************************************] 100
N count     	np          	cp          
2048        	0.506053686 	0.942011356 
4096        	1.577385187 	0.009101868 
8192        	16.427660465	0.033759832 
10000       	50.184625864	0.253522158 


### Задание 4.

Ряд Маклорена.

На вход подается массив размерностью $N$ со значениями [-10, 10] и число $M$ - количество многочленов для апрокимации. Необходимо реализовать функцию, которая вычисляет разложение функции $sin(x)$ в ряд Маклорена для заданного массива и количество многочленов для апрокимации. Для экспериментов ограничим значение, которое может принимать $N$ в задачах следующим интервалом [1, 10000000], а $M$ - одним, любым значением, но не меньше 5.

In [7]:
def compete4(M_mas, lib_mas, func_mas, step = 1):
    mmas = []
    for i, M in enumerate(M_mas):
        mmas.append([])
        for j, lib in enumerate(lib_mas):
            pretty_load(i*len(lib_mas) + j, len(M_mas) * len(lib_mas))

            arr1 = lib.arange(-10, 10+step, step)
            start = time.time()
            result = func_mas[j](arr1, M) 
            # print ("\n", result)

            mmas[-1].append(time.time() - start)
    pretty_load(len(M_mas) * len(lib_mas), len(N_mas) * len(lib_mas))
    return mmas


def func4_np(x, M):
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    if M <= 0:
        return result

    term = x.copy()
    result += term
    x2 = x * x

    for k in range(1, M):
        term *= -x2         
        term /= (2 * k) * (2 * k + 1)  
        result += term
    return result

def func4_cp(x, M):
    x = cp.asarray(x, dtype=float)
    result = cp.zeros_like(x)
    if M <= 0:
        return result

    term = x.copy()
    result += term
    x2 = x * x

    for k in range(1, M):
        term *= -x2         
        term /= (2 * k) * (2 * k + 1)  
        result += term
    return result


M_mas = [300, 1000, 2000]
res = compete4(M_mas, [np, cp], [func4_np, func4_cp], step = 0.0001)

print(f"""\n{"N count":<12}\t{"np":<12}\t{"cp":<12}""")
for i, row in enumerate(res):
    print(f"{M_mas[i]:<12}\t{row[0]:<12.9f}\t{row[1]:<12.9f}")

[**************************************            ] 75
N count     	np          	cp          
300         	0.321179628 	0.084025383 
1000        	0.203146696 	0.115360498 
2000        	0.404279947 	0.282917976 
